# CartFlow (P06) — Week 7: Gold Model, KPI Implementation & Reconciliation

**Working artifact:** `notebooks/05_gold_aggregations.ipynb`

This notebook converts the accepted **Trusted Silver** layer into governed CartFlow Gold dimensions, facts, batch summaries and KPI outputs.

**Week-7 boundary:** Gold model + KPI logic + reconciliation only. Power BI export/modeling is a Week-8 activity, and streaming is a later activity.

**Core engineering rule:** aggregate item and payment children independently to `order_id`, reconcile money within **INR 0.05**, then join each summary exactly once to `fact_order`. Never raw-join items, payments, reviews and events together.


## 1. Week-7 contract and expected outcome

The approved CartFlow playbook defines Week 7 as **Gold Model, KPI Implementation and Reconciliation**. The required build flow is:

1. `dim_date`, `dim_seller`, `dim_product_category`, `dim_customer_segment`, `dim_payment_method`, `dim_order_status`
2. `fact_order_item` at `order_item_id`; independently aggregate item value/freight to `order_id`
3. Independently aggregate payments to `order_id` and reconcile within INR 0.05 before `fact_order`
4. Build `fact_order`, `fact_payment`, `fact_review`, and `fact_order_status_event` at their governed grains
5. Build five batch summaries and implement eight KPI contracts with explicit denominators/exclusions

**Expected Week-7 state:** six dimensions, five facts, five batch summaries and eight KPI contracts implemented from Trusted Silver with controlled child aggregation.

No execution result is pre-filled in this notebook. All counts, PASS/CHECK results and histories must come from the user's Databricks run.


## 2. Gold object catalog

### Dimensions
- `dim_date` — one row per date
- `dim_seller` — one row per seller
- `dim_product_category` — one row per product/category combination available in Trusted Silver
- `dim_customer_segment` — one row per customer segment
- `dim_payment_method` — one row per payment method
- `dim_order_status` — one row per order-status / delivery-band combination

### Facts
- `fact_order_item` — one trusted item row per `order_item_id`
- `fact_order` — one trusted order row per `order_id`
- `fact_payment` — one trusted payment/installment row per `payment_id`
- `fact_review` — one trusted eligible review row per `review_id`
- `fact_order_status_event` — one derived lifecycle transition per order/timestamp/status event

### Batch summaries
- `agg_sales_daily`
- `agg_seller_performance`
- `agg_category_sales`
- `agg_delivery_delay`
- `agg_payment_review`

The approved playbook also names `agg_live_order_status` for the later live-order reporting path; it is **not a Week-7 batch summary** and is intentionally not built here.


## 3. Week-6 handoff — trusted inputs only

Gold must consume accepted Trusted Silver outputs. It must not read Silver Candidate or Quarantine.

The Week-6 handoff for this CartFlow implementation is:

- `trusted_silver_orders`
- `trusted_silver_order_items`
- `trusted_silver_payments`
- `trusted_silver_reviews`
- `trusted_silver_sellers`

The Week-6 notebook proves Candidate = Trusted + Quarantine at physical-record grain and zero Trusted/Quarantine overlap. Resolve any Week-6 CHECK before proceeding.


In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;

In [ ]:
%sql
SHOW TABLES LIKE 'trusted_silver_*';

In [ ]:
%sql
SELECT 'orders' AS entity, COUNT(*) AS trusted_rows FROM trusted_silver_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM trusted_silver_order_items
UNION ALL SELECT 'payments', COUNT(*) FROM trusted_silver_payments
UNION ALL SELECT 'reviews', COUNT(*) FROM trusted_silver_reviews
UNION ALL SELECT 'sellers', COUNT(*) FROM trusted_silver_sellers
ORDER BY entity;

**Checkpoint:** the five Trusted Silver tables must exist in the same catalog/schema used by Week 6. Do not substitute Candidate or Quarantine tables.

## 4. KPI register — all eight approved contracts

| KPI | Formula / numerator | Denominator | Exclusions / handling | Gold source |
|---|---|---|---|---|
| Total Orders | `COUNT(DISTINCT order_id)` | — | Trusted eligible orders in created/purchase scope | `fact_order` / `agg_sales_daily` |
| GMV | `SUM(item_price)` | — | Trusted item rows whose parent order is not cancelled; freight is not GMV | `fact_order_item` / summaries |
| Average Order Value | GMV | distinct non-cancelled trusted orders | Same scope/filter context; BLANK when denominator is zero | `fact_order` + `fact_order_item` / `agg_sales_daily` |
| Delivered Order Rate | distinct delivered/returned trusted orders | distinct trusted orders created in scope | Order grain only; not item count; BLANK at zero denominator | `fact_order` / `agg_sales_daily` |
| On-Time Delivery Rate | delivered/returned orders with `delivered_ts <= estimated_delivery_ts` | delivered/returned orders with both timestamps | Exclude undelivered and missing timestamp pairs; BLANK at zero | `fact_order` / `agg_delivery_delay` |
| Cancellation Rate | distinct cancelled trusted orders | distinct trusted orders created in scope | Order grain only; BLANK at zero | `fact_order` / `agg_sales_daily` |
| Payment Reconciliation Rate | eligible orders with `ABS(payment_total-order_total) <= 0.05` | trusted orders with payment rows and complete child coverage | Exclude incomplete/quarantined child coverage; BLANK at zero | `fact_order` / `agg_payment_review` |
| Average Review Score | `AVG(review_score)` | — | Trusted reviews for delivered/returned eligible orders; BLANK if none | `fact_review` / `agg_payment_review` |

The payment reconciliation tolerance is **INR 0.05**, and zero denominators must return BLANK rather than zero/error where the contract specifies it.


## 5. Build dimensions

These are reporting-context dimensions only. They do not alter Trusted Silver trust status. Each object is rebuilt deterministically from Trusted Silver for the current approved snapshot.


In [ ]:
%sql
CREATE OR REPLACE TABLE dim_date
USING DELTA AS
WITH bounds AS (
  SELECT
    MIN(to_date(purchase_ts)) AS min_date,
    MAX(to_date(purchase_ts)) AS max_date
  FROM trusted_silver_orders
  WHERE purchase_ts IS NOT NULL
)
SELECT
  date_value AS date_key,
  year(date_value) AS calendar_year,
  quarter(date_value) AS calendar_quarter,
  month(date_value) AS calendar_month,
  weekofyear(date_value) AS calendar_week,
  date_format(date_value, 'yyyy-MM') AS year_month,
  date_format(date_value, 'yyyy-MM-dd') AS date_label,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM bounds
LATERAL VIEW explode(sequence(min_date, max_date, interval 1 day)) d AS date_value
WHERE min_date IS NOT NULL AND max_date IS NOT NULL;

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_seller
USING DELTA AS
SELECT
  seller_id,
  seller_region,
  seller_state,
  seller_type,
  service_band,
  seller_status,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_sellers
QUALIFY ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY active_from DESC, active_to DESC) = 1;

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_product_category
USING DELTA AS
SELECT DISTINCT
  product_id,
  category_code,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_order_items
WHERE product_id IS NOT NULL OR category_code IS NOT NULL;

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_customer_segment
USING DELTA AS
SELECT DISTINCT
  customer_segment,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_orders
WHERE customer_segment IS NOT NULL;

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_payment_method
USING DELTA AS
SELECT DISTINCT
  payment_method,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_payments
WHERE payment_method IS NOT NULL;

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_order_status
USING DELTA AS
SELECT DISTINCT
  order_status,
  delivery_band,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_orders;

### Dimension proof

The approved validation method is count/grain plus primary/business-key uniqueness. A duplicate dimension key can multiply downstream facts, so do not hide duplicates with `DISTINCT` after an unsafe join.


In [ ]:
%sql
SELECT 'dim_date' AS object_name, COUNT(*) AS rows, COUNT(DISTINCT date_key) AS distinct_keys
FROM dim_date
UNION ALL
SELECT 'dim_seller', COUNT(*), COUNT(DISTINCT seller_id) FROM dim_seller
UNION ALL
SELECT 'dim_product_category', COUNT(*), COUNT(DISTINCT product_id || '|' || COALESCE(category_code,'')) FROM dim_product_category
UNION ALL
SELECT 'dim_customer_segment', COUNT(*), COUNT(DISTINCT customer_segment) FROM dim_customer_segment
UNION ALL
SELECT 'dim_payment_method', COUNT(*), COUNT(DISTINCT payment_method) FROM dim_payment_method
UNION ALL
SELECT 'dim_order_status', COUNT(*), COUNT(DISTINCT COALESCE(order_status,'') || '|' || COALESCE(delivery_band,'')) FROM dim_order_status;

In [ ]:
%sql
SELECT 'dim_seller' AS dimension, seller_id AS key_value, COUNT(*) AS occurrences
FROM dim_seller GROUP BY seller_id HAVING COUNT(*) > 1
UNION ALL
SELECT 'dim_customer_segment', customer_segment, COUNT(*)
FROM dim_customer_segment GROUP BY customer_segment HAVING COUNT(*) > 1
UNION ALL
SELECT 'dim_payment_method', payment_method, COUNT(*)
FROM dim_payment_method GROUP BY payment_method HAVING COUNT(*) > 1;

## 6. Build `fact_order_item` at exact item grain

The playbook requires `fact_order_item` at `order_item_id`. Item and freight totals are aggregated separately to `order_id` before any order-level join.


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_order_item
USING DELTA AS
SELECT
  source_record_id,
  order_item_id,
  order_id,
  order_item_seq,
  product_id,
  category_code,
  seller_id,
  item_price,
  freight_value,
  quantity,
  item_total,
  freight_share,
  item_created_ts,
  _source_file_name,
  _source_file_path,
  _ingested_at,
  _ingestion_run_id,
  _bronze_schema_version,
  _bronze_record_hash,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_order_items;

In [ ]:
%sql
SELECT
  COUNT(*) AS fact_rows,
  COUNT(DISTINCT order_item_id) AS distinct_order_item_ids,
  SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS null_order_item_ids,
  SUM(CASE WHEN item_price IS NULL OR item_price < 0 THEN 1 ELSE 0 END) AS invalid_item_price,
  SUM(CASE WHEN freight_value IS NULL OR freight_value < 0 THEN 1 ELSE 0 END) AS invalid_freight
FROM fact_order_item;

In [ ]:
%sql
SELECT order_item_id, COUNT(*) AS occurrences
FROM fact_order_item
GROUP BY order_item_id
HAVING COUNT(*) > 1;

## 7. Independently aggregate item children to order grain

`item_order_summary` is intentionally one row per `order_id`. This prevents item rows from multiplying payment rows.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW item_order_summary AS
SELECT
  order_id,
  SUM(item_price) AS gmv,
  SUM(item_total) AS item_total,
  SUM(freight_value) AS freight_total,
  COUNT(*) AS item_line_count,
  COUNT(DISTINCT order_item_id) AS distinct_item_count
FROM fact_order_item
GROUP BY order_id;

In [ ]:
%sql
SELECT COUNT(*) AS order_summary_rows,
       COUNT(DISTINCT order_id) AS distinct_order_ids,
       SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_ids
FROM item_order_summary;

## 8. Build `fact_payment` and independently aggregate payments

The payment fact stays at installment/payment-row grain. The order summary is built separately before `fact_order`.


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_payment
USING DELTA AS
SELECT
  source_record_id,
  payment_id,
  order_id,
  installment_no,
  payment_method,
  payment_value,
  payment_ts,
  currency_code,
  _source_file_name,
  _source_file_path,
  _ingested_at,
  _ingestion_run_id,
  _bronze_schema_version,
  _bronze_record_hash,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_payments;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payment_order_summary AS
SELECT
  order_id,
  SUM(payment_value) AS payment_total,
  COUNT(*) AS payment_row_count,
  COUNT(DISTINCT payment_id) AS distinct_payment_count,
  MIN(installment_no) AS min_installment_no,
  MAX(installment_no) AS max_installment_no
FROM fact_payment
GROUP BY order_id;

In [ ]:
%sql
SELECT
  COUNT(*) AS fact_payment_rows,
  COUNT(DISTINCT payment_id) AS distinct_payment_ids,
  SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END) AS null_payment_ids,
  SUM(CASE WHEN payment_value IS NULL OR payment_value < 0 THEN 1 ELSE 0 END) AS invalid_payment_values
FROM fact_payment;

In [ ]:
%sql
SELECT payment_id, COUNT(*) AS occurrences
FROM fact_payment
GROUP BY payment_id
HAVING COUNT(*) > 1;

## 9. Payment/item reconciliation before `fact_order`

The governed rule is to compare independently aggregated order-level money. The approved tolerance is INR 0.05.

For this implementation, `order_total = item GMV + freight`, while **GMV itself excludes freight**.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW order_money_reconciliation AS
SELECT
  COALESCE(i.order_id, p.order_id) AS order_id,
  i.gmv,
  i.freight_total,
  i.item_total,
  p.payment_total,
  CASE WHEN i.order_id IS NOT NULL AND p.order_id IS NOT NULL THEN 1 ELSE 0 END AS complete_child_coverage,
  CASE
    WHEN i.order_id IS NOT NULL AND p.order_id IS NOT NULL
    THEN ABS(COALESCE(p.payment_total,0) - (COALESCE(i.gmv,0) + COALESCE(i.freight_total,0)))
  END AS payment_order_difference,
  CASE
    WHEN i.order_id IS NOT NULL
     AND p.order_id IS NOT NULL
     AND ABS(COALESCE(p.payment_total,0) - (COALESCE(i.gmv,0) + COALESCE(i.freight_total,0))) <= 0.05
    THEN 'RECONCILED'
    WHEN i.order_id IS NOT NULL AND p.order_id IS NOT NULL THEN 'NOT_RECONCILED'
    ELSE 'INCOMPLETE_CHILD_COVERAGE'
  END AS reconciliation_status
FROM item_order_summary i
FULL OUTER JOIN payment_order_summary p
  ON i.order_id = p.order_id;

In [ ]:
%sql
SELECT reconciliation_status, COUNT(*) AS order_count
FROM order_money_reconciliation
GROUP BY reconciliation_status
ORDER BY reconciliation_status;

**Important:** an order with missing item/payment child coverage is not treated as reconciled. This prevents incomplete or quarantined child populations from being counted as successful payment reconciliation.

## 10. Build `fact_order` at one row per trusted order

Only one row per `order_id` is allowed. Child summaries are joined after independent aggregation.

The order fact carries order lifecycle fields plus the order-level item/payment measures needed by downstream KPIs.


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_order
USING DELTA AS
SELECT
  o.source_record_id,
  o.order_id,
  o.customer_region,
  o.customer_state,
  o.customer_city_code,
  o.customer_segment,
  o.order_status,
  o.purchase_ts,
  o.approval_ts,
  o.carrier_handoff_ts,
  o.delivered_ts,
  o.estimated_delivery_ts,
  o.return_ts,
  o.currency_code,
  o.delivery_days,
  o.delay_days,
  o.delivery_band,
  i.gmv,
  i.item_total,
  i.freight_total,
  i.item_line_count,
  p.payment_total,
  p.payment_row_count,
  p.distinct_payment_count,
  r.reconciliation_status,
  r.payment_order_difference,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_orders o
LEFT JOIN item_order_summary i
  ON o.order_id = i.order_id
LEFT JOIN payment_order_summary p
  ON o.order_id = p.order_id
LEFT JOIN order_money_reconciliation r
  ON o.order_id = r.order_id;

In [ ]:
%sql
SELECT
  COUNT(*) AS fact_order_rows,
  COUNT(DISTINCT order_id) AS distinct_order_ids,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_ids,
  SUM(CASE WHEN gmv IS NOT NULL AND gmv < 0 THEN 1 ELSE 0 END) AS negative_gmv
FROM fact_order;

In [ ]:
%sql
SELECT order_id, COUNT(*) AS occurrences
FROM fact_order
GROUP BY order_id
HAVING COUNT(*) > 1;

## 11. Fact-order money reconciliation proof

The fact-level values must agree with the independently calculated child summaries. This is a fan-out guardrail, not a raw multi-child join.


In [ ]:
%sql
WITH checks AS (
  SELECT
    (SELECT SUM(gmv) FROM item_order_summary) AS item_summary_gmv,
    (SELECT SUM(gmv) FROM fact_order WHERE gmv IS NOT NULL) AS fact_order_gmv,
    (SELECT SUM(payment_total) FROM payment_order_summary) AS payment_summary_total,
    (SELECT SUM(payment_total) FROM fact_order WHERE payment_total IS NOT NULL) AS fact_order_payment_total
)
SELECT *,
       ABS(item_summary_gmv - fact_order_gmv) AS gmv_difference,
       ABS(payment_summary_total - fact_order_payment_total) AS payment_difference,
       CASE WHEN ABS(item_summary_gmv - fact_order_gmv) <= 0.05
                  AND ABS(payment_summary_total - fact_order_payment_total) <= 0.05
            THEN 'PASS' ELSE 'CHECK' END AS status
FROM checks;

## 12. Build `fact_review` at exact review grain

Only trusted reviews for delivered/returned orders are eligible for the Average Review Score KPI. The review fact retains the review record at `review_id` grain and exposes the parent order status.


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_review
USING DELTA AS
SELECT
  r.source_record_id,
  r.review_id,
  r.order_id,
  r.review_score,
  r.review_date,
  r.review_sentiment,
  o.order_status,
  CASE WHEN o.order_status IN ('delivered','returned') THEN TRUE ELSE FALSE END AS eligible_fulfilled_order,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_reviews r
LEFT JOIN fact_order o
  ON r.order_id = o.order_id;

In [ ]:
%sql
SELECT
  COUNT(*) AS fact_review_rows,
  COUNT(DISTINCT review_id) AS distinct_review_ids,
  SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS null_review_ids,
  SUM(CASE WHEN review_score IS NULL OR review_score NOT BETWEEN 1 AND 5 THEN 1 ELSE 0 END) AS invalid_scores,
  SUM(CASE WHEN eligible_fulfilled_order THEN 1 ELSE 0 END) AS eligible_reviews
FROM fact_review;

In [ ]:
%sql
SELECT review_id, COUNT(*) AS occurrences
FROM fact_review
GROUP BY review_id
HAVING COUNT(*) > 1;

## 13. Build `fact_order_status_event`

The Week-7 playbook requires a fifth fact at exact grain, while the actual incremental order-status event drops are introduced later in the project. To avoid inventing streaming records, this Week-7 notebook builds a **batch lifecycle-transition fact derived only from trusted order lifecycle timestamps**.

It is not the Week-10 streaming implementation. When the governed streaming event source is introduced, its event-grain implementation should replace/extend this object according to the later streaming contract.


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_order_status_event
USING DELTA AS
SELECT
  order_id,
  'purchase' AS event_status,
  purchase_ts AS event_ts,
  1 AS event_sequence,
  source_record_id,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM trusted_silver_orders
WHERE purchase_ts IS NOT NULL

UNION ALL
SELECT
  order_id, 'approval', approval_ts, 2, source_record_id,
  current_timestamp(), 'cartflow_gold_v1.0'
FROM trusted_silver_orders
WHERE approval_ts IS NOT NULL

UNION ALL
SELECT
  order_id, 'carrier_handoff', carrier_handoff_ts, 3, source_record_id,
  current_timestamp(), 'cartflow_gold_v1.0'
FROM trusted_silver_orders
WHERE carrier_handoff_ts IS NOT NULL

UNION ALL
SELECT
  order_id, 'delivered', delivered_ts, 4, source_record_id,
  current_timestamp(), 'cartflow_gold_v1.0'
FROM trusted_silver_orders
WHERE delivered_ts IS NOT NULL

UNION ALL
SELECT
  order_id, 'returned', return_ts, 5, source_record_id,
  current_timestamp(), 'cartflow_gold_v1.0'
FROM trusted_silver_orders
WHERE return_ts IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS fact_status_event_rows,
  COUNT(DISTINCT order_id) AS distinct_orders_with_events,
  SUM(CASE WHEN event_ts IS NULL THEN 1 ELSE 0 END) AS null_event_timestamps
FROM fact_order_status_event;

## 14. Build the five approved batch summaries

The summaries are designed around the five batch-summary objects named in the CartFlow Gold model. They are derived from the governed facts and preserve the anti-fan-out pattern.

`agg_live_order_status` is deliberately excluded because the live status feed is a later streaming dependency.


In [ ]:
%sql
CREATE OR REPLACE TABLE agg_sales_daily
USING DELTA AS
SELECT
  to_date(purchase_ts) AS sales_date,
  COUNT(DISTINCT order_id) AS total_orders,
  COUNT(DISTINCT CASE WHEN order_status <> 'cancelled' THEN order_id END) AS non_cancelled_orders,
  SUM(CASE WHEN order_status <> 'cancelled' THEN COALESCE(gmv,0) ELSE 0 END) AS gmv,
  SUM(CASE WHEN order_status <> 'cancelled' THEN COALESCE(freight_total,0) ELSE 0 END) AS freight,
  CASE
    WHEN COUNT(DISTINCT CASE WHEN order_status <> 'cancelled' THEN order_id END) = 0 THEN NULL
    ELSE SUM(CASE WHEN order_status <> 'cancelled' THEN COALESCE(gmv,0) ELSE 0 END)
         / COUNT(DISTINCT CASE WHEN order_status <> 'cancelled' THEN order_id END)
  END AS aov,
  COUNT(*) AS trusted_order_rows,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM fact_order
WHERE purchase_ts IS NOT NULL
GROUP BY to_date(purchase_ts);

In [ ]:
%sql
CREATE OR REPLACE TABLE agg_seller_performance
USING DELTA AS
WITH seller_sales AS (
  SELECT
    foi.seller_id,
    fo.order_id,
    fo.order_status,
    fo.purchase_ts,
    foi.item_price
  FROM fact_order_item foi
  JOIN fact_order fo
    ON foi.order_id = fo.order_id
),
seller_reviews AS (
  SELECT
    soi.seller_id,
    AVG(fr.review_score) AS average_review_score
  FROM fact_review fr
  JOIN (
    SELECT DISTINCT order_id, seller_id
    FROM fact_order_item
  ) soi
    ON fr.order_id = soi.order_id
  WHERE fr.eligible_fulfilled_order = TRUE
  GROUP BY soi.seller_id
)
SELECT
  s.seller_id,
  COUNT(DISTINCT s.order_id) AS order_count,
  COUNT(DISTINCT CASE WHEN s.order_status = 'cancelled' THEN s.order_id END) AS cancelled_orders,
  COUNT(DISTINCT CASE WHEN s.order_status = 'returned' THEN s.order_id END) AS returned_orders,
  SUM(CASE WHEN s.order_status <> 'cancelled' THEN s.item_price ELSE 0 END) AS gmv,
  sr.average_review_score,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM seller_sales s
LEFT JOIN seller_reviews sr
  ON s.seller_id = sr.seller_id
GROUP BY s.seller_id, sr.average_review_score;

In [ ]:
%sql
CREATE OR REPLACE TABLE agg_category_sales
USING DELTA AS
WITH category_item AS (
  SELECT
    foi.category_code,
    foi.order_id,
    foi.item_price,
    foi.item_total,
    foi.freight_value,
    fo.order_status,
    fo.purchase_ts
  FROM fact_order_item foi
  JOIN fact_order fo ON foi.order_id = fo.order_id
)
SELECT
  date_format(to_date(purchase_ts), 'yyyy-MM') AS year_month,
  category_code,
  COUNT(*) AS item_lines,
  COUNT(DISTINCT order_id) AS order_count,
  SUM(CASE WHEN order_status <> 'cancelled' THEN item_price ELSE 0 END) AS gmv,
  SUM(CASE WHEN order_status <> 'cancelled' THEN item_total ELSE 0 END) AS item_total,
  SUM(CASE WHEN order_status <> 'cancelled' THEN freight_value ELSE 0 END) AS freight,
  CASE
    WHEN COUNT(DISTINCT CASE WHEN order_status <> 'cancelled' THEN order_id END) = 0 THEN NULL
    ELSE SUM(CASE WHEN order_status <> 'cancelled' THEN item_price ELSE 0 END)
         / COUNT(DISTINCT CASE WHEN order_status <> 'cancelled' THEN order_id END)
  END AS average_order_value,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM category_item
WHERE purchase_ts IS NOT NULL
GROUP BY date_format(to_date(purchase_ts), 'yyyy-MM'), category_code;

In [ ]:
%sql
CREATE OR REPLACE TABLE agg_delivery_delay
USING DELTA AS
SELECT
  to_date(purchase_ts) AS purchase_date,
  customer_state,
  COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned')
                       AND delivered_ts IS NOT NULL
                       AND estimated_delivery_ts IS NOT NULL
                      THEN order_id END) AS delivery_eligible_orders,
  COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned')
                       AND delivered_ts IS NOT NULL
                       AND estimated_delivery_ts IS NOT NULL
                       AND delivered_ts <= estimated_delivery_ts
                      THEN order_id END) AS on_time_orders,
  AVG(CASE WHEN order_status IN ('delivered','returned')
                AND delivered_ts IS NOT NULL
                AND estimated_delivery_ts IS NOT NULL
           THEN delay_days END) AS average_delay_days,
  CASE
    WHEN COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned')
                              AND delivered_ts IS NOT NULL
                              AND estimated_delivery_ts IS NOT NULL
                             THEN order_id END) = 0 THEN NULL
    ELSE COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned')
                              AND delivered_ts IS NOT NULL
                              AND estimated_delivery_ts IS NOT NULL
                              AND delivered_ts <= estimated_delivery_ts
                             THEN order_id END)
         / COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned')
                                AND delivered_ts IS NOT NULL
                                AND estimated_delivery_ts IS NOT NULL
                               THEN order_id END)
  END AS on_time_delivery_rate,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM fact_order
WHERE purchase_ts IS NOT NULL
GROUP BY to_date(purchase_ts), customer_state;

In [ ]:
%sql
CREATE OR REPLACE TABLE agg_payment_review
USING DELTA AS
WITH payment_stats AS (
  SELECT
    fo.currency_code,
    COUNT(DISTINCT CASE
      WHEN p.payment_rows > 0
       AND i.item_rows > 0
       AND ABS(p.payment_total - (i.gmv + i.freight_total)) <= 0.05
      THEN fo.order_id END) AS reconciled_orders,
    COUNT(DISTINCT CASE
      WHEN p.payment_rows > 0 AND i.item_rows > 0
      THEN fo.order_id END) AS complete_coverage_orders
  FROM fact_order fo
  LEFT JOIN (
    SELECT order_id, SUM(item_price) AS gmv, SUM(freight_value) AS freight_total, COUNT(*) AS item_rows
    FROM fact_order_item GROUP BY order_id
  ) i ON fo.order_id = i.order_id
  LEFT JOIN (
    SELECT order_id, SUM(payment_value) AS payment_total, COUNT(*) AS payment_rows
    FROM fact_payment GROUP BY order_id
  ) p ON fo.order_id = p.order_id
  GROUP BY fo.currency_code
),
review_stats AS (
  SELECT
    fo.currency_code,
    AVG(fr.review_score) AS average_review_score,
    COUNT(DISTINCT fr.order_id) AS reviewed_eligible_orders
  FROM fact_review fr
  JOIN fact_order fo ON fr.order_id = fo.order_id
  WHERE fr.eligible_fulfilled_order = TRUE
  GROUP BY fo.currency_code
),
fulfilled_stats AS (
  SELECT
    currency_code,
    COUNT(DISTINCT order_id) AS eligible_fulfilled_orders
  FROM fact_order
  WHERE order_status IN ('delivered','returned')
  GROUP BY currency_code
)
SELECT
  p.currency_code,
  p.reconciled_orders,
  p.complete_coverage_orders,
  CASE
    WHEN p.complete_coverage_orders = 0 THEN NULL
    ELSE p.reconciled_orders * 1.0 / p.complete_coverage_orders
  END AS payment_reconciliation_rate,
  r.average_review_score,
  COALESCE(r.reviewed_eligible_orders, 0) AS reviewed_eligible_orders,
  COALESCE(f.eligible_fulfilled_orders, 0) AS eligible_fulfilled_orders,
  CASE
    WHEN COALESCE(f.eligible_fulfilled_orders, 0) = 0 THEN NULL
    ELSE COALESCE(r.reviewed_eligible_orders, 0) * 1.0 / f.eligible_fulfilled_orders
  END AS reviewed_order_coverage,
  current_timestamp() AS _gold_created_at,
  'cartflow_gold_v1.0' AS _gold_schema_version
FROM payment_stats p
LEFT JOIN review_stats r ON p.currency_code <=> r.currency_code
LEFT JOIN fulfilled_stats f ON p.currency_code <=> f.currency_code;

## 15. Eight KPI implementations

The KPI output below is a governed validation/serving view. It deliberately returns NULL for zero-denominator rate KPIs rather than converting an undefined rate to 0.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW cartflow_week07_kpis AS
WITH order_scope AS (
  SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT CASE WHEN order_status <> 'cancelled' THEN order_id END) AS non_cancelled_orders,
    COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned') THEN order_id END) AS delivered_or_returned_orders,
    COUNT(DISTINCT CASE WHEN order_status = 'cancelled' THEN order_id END) AS cancelled_orders
  FROM fact_order
  WHERE purchase_ts IS NOT NULL
),
gmv_scope AS (
  SELECT
    SUM(CASE WHEN fo.order_status <> 'cancelled' THEN foi.item_price ELSE 0 END) AS gmv
  FROM fact_order_item foi
  JOIN fact_order fo ON foi.order_id = fo.order_id
  WHERE fo.purchase_ts IS NOT NULL
),
delivery_scope AS (
  SELECT
    COUNT(DISTINCT CASE
      WHEN order_status IN ('delivered','returned')
       AND delivered_ts IS NOT NULL
       AND estimated_delivery_ts IS NOT NULL
      THEN order_id END) AS delivery_denominator,
    COUNT(DISTINCT CASE
      WHEN order_status IN ('delivered','returned')
       AND delivered_ts IS NOT NULL
       AND estimated_delivery_ts IS NOT NULL
       AND delivered_ts <= estimated_delivery_ts
      THEN order_id END) AS on_time_numerator
  FROM fact_order
  WHERE purchase_ts IS NOT NULL
),
payment_scope AS (
  SELECT
    COUNT(DISTINCT CASE
      WHEN p.payment_rows > 0
       AND i.item_rows > 0
       AND ABS(p.payment_total - (i.gmv + i.freight_total)) <= 0.05
      THEN fo.order_id END) AS reconciled_numerator,
    COUNT(DISTINCT CASE
      WHEN p.payment_rows > 0 AND i.item_rows > 0
      THEN fo.order_id END) AS reconciliation_denominator
  FROM fact_order fo
  LEFT JOIN (
    SELECT order_id, SUM(item_price) AS gmv, SUM(freight_value) AS freight_total, COUNT(*) AS item_rows
    FROM fact_order_item GROUP BY order_id
  ) i ON fo.order_id = i.order_id
  LEFT JOIN (
    SELECT order_id, SUM(payment_value) AS payment_total, COUNT(*) AS payment_rows
    FROM fact_payment GROUP BY order_id
  ) p ON fo.order_id = p.order_id
  WHERE fo.purchase_ts IS NOT NULL
),
review_scope AS (
  SELECT
    AVG(review_score) AS average_review_score,
    COUNT(*) AS eligible_review_rows
  FROM fact_review
  WHERE eligible_fulfilled_order = TRUE
)
SELECT 'Total Orders' AS kpi,
       CAST(o.total_orders AS DECIMAL(18,4)) AS value,
       'COUNT(DISTINCT order_id)' AS formula
FROM order_scope o
UNION ALL
SELECT 'Gross Merchandise Value (GMV)',
       CAST(g.gmv AS DECIMAL(18,4)),
       'SUM(item_price) for non-cancelled trusted order items'
FROM gmv_scope g
UNION ALL
SELECT 'Average Order Value',
       CASE WHEN o.non_cancelled_orders = 0 THEN NULL
            ELSE CAST(g.gmv / o.non_cancelled_orders AS DECIMAL(18,4)) END,
       'GMV / distinct non-cancelled trusted orders'
FROM order_scope o CROSS JOIN gmv_scope g
UNION ALL
SELECT 'Delivered Order Rate',
       CASE WHEN o.total_orders = 0 THEN NULL
            ELSE CAST(o.delivered_or_returned_orders * 1.0 / o.total_orders AS DECIMAL(18,6)) END,
       'distinct delivered/returned trusted orders / distinct trusted orders'
FROM order_scope o
UNION ALL
SELECT 'On-Time Delivery Rate',
       CASE WHEN d.delivery_denominator = 0 THEN NULL
            ELSE CAST(d.on_time_numerator * 1.0 / d.delivery_denominator AS DECIMAL(18,6)) END,
       'delivered/returned on-time orders / delivered/returned orders with both timestamps'
FROM delivery_scope d
UNION ALL
SELECT 'Cancellation Rate',
       CASE WHEN o.total_orders = 0 THEN NULL
            ELSE CAST(o.cancelled_orders * 1.0 / o.total_orders AS DECIMAL(18,6)) END,
       'distinct cancelled trusted orders / distinct trusted orders'
FROM order_scope o
UNION ALL
SELECT 'Payment Reconciliation Rate',
       CASE WHEN p.reconciliation_denominator = 0 THEN NULL
            ELSE CAST(p.reconciled_numerator * 1.0 / p.reconciliation_denominator AS DECIMAL(18,6)) END,
       'reconciled eligible orders / trusted orders with complete item+payment coverage'
FROM payment_scope p
UNION ALL
SELECT 'Average Review Score',
       CASE WHEN r.eligible_review_rows = 0 THEN NULL
            ELSE CAST(r.average_review_score AS DECIMAL(18,4)) END,
       'AVG(review_score) over trusted eligible fulfilled-order reviews'
FROM review_scope r;

In [ ]:
%sql
SELECT * FROM cartflow_week07_kpis ORDER BY kpi;

## 16. KPI spot checks

Use these checks to manually validate the most important numerator/denominator relationships before handing Gold to Week 8.


In [ ]:
%sql
SELECT
  SUM(CASE WHEN order_status IN ('delivered','returned') THEN 1 ELSE 0 END) AS delivered_returned_order_rows,
  COUNT(DISTINCT CASE WHEN order_status IN ('delivered','returned') THEN order_id END) AS delivered_returned_distinct_orders,
  COUNT(DISTINCT order_id) AS trusted_distinct_orders
FROM fact_order
WHERE purchase_ts IS NOT NULL;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN order_status <> 'cancelled' THEN gmv ELSE 0 END) AS fact_order_gmv,
  (SELECT SUM(CASE WHEN fo.order_status <> 'cancelled' THEN foi.item_price ELSE 0 END)
   FROM fact_order_item foi
   JOIN fact_order fo ON foi.order_id = fo.order_id) AS independently_aggregated_gmv;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN payment_order_difference <= 0.05 AND reconciliation_status = 'RECONCILED' THEN 1 ELSE 0 END) AS reconciled_orders,
  SUM(CASE WHEN reconciliation_status <> 'INCOMPLETE_CHILD_COVERAGE' THEN 1 ELSE 0 END) AS complete_coverage_orders
FROM fact_order;

## 17. Gold grain and scope proof

The approved Week-7 validation method requires distinct physical/business keys, approved grain uniqueness, no unexplained row loss/duplication, lineage, dependency checks and money reconciliation.


In [ ]:
%sql
SELECT 'fact_order' AS object_name,
       COUNT(*) AS rows,
       COUNT(DISTINCT order_id) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM fact_order
UNION ALL
SELECT 'fact_order_item',
       COUNT(*),
       COUNT(DISTINCT order_item_id),
       SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END)
FROM fact_order_item
UNION ALL
SELECT 'fact_payment',
       COUNT(*),
       COUNT(DISTINCT payment_id),
       SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END)
FROM fact_payment
UNION ALL
SELECT 'fact_review',
       COUNT(*),
       COUNT(DISTINCT review_id),
       SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END)
FROM fact_review;

In [ ]:
%sql
SELECT
  COUNT(*) AS trusted_order_rows,
  COUNT(DISTINCT order_id) AS trusted_distinct_order_ids,
  (SELECT COUNT(*) FROM fact_order) AS fact_order_rows,
  (SELECT COUNT(DISTINCT order_id) FROM fact_order) AS fact_order_distinct_order_ids,
  CASE
    WHEN COUNT(*) = (SELECT COUNT(*) FROM fact_order)
     AND COUNT(DISTINCT order_id) = (SELECT COUNT(DISTINCT order_id) FROM fact_order)
    THEN 'PASS' ELSE 'CHECK'
  END AS order_scope_status
FROM trusted_silver_orders;

## 18. Lineage spot check

Trace an anchor order through Trusted Silver → Gold fact → item/payment child facts. Use a real `order_id` from your Databricks run; this cell does not hard-code an expected project answer.


In [ ]:
%sql
SELECT
  o.order_id,
  o.source_record_id,
  o.order_status,
  o.purchase_ts,
  o.gmv,
  o.payment_total,
  o.reconciliation_status,
  COUNT(DISTINCT i.order_item_id) AS item_rows,
  COUNT(DISTINCT p.payment_id) AS payment_rows
FROM fact_order o
LEFT JOIN fact_order_item i ON o.order_id = i.order_id
LEFT JOIN fact_payment p ON o.order_id = p.order_id
GROUP BY
  o.order_id, o.source_record_id, o.order_status, o.purchase_ts,
  o.gmv, o.payment_total, o.reconciliation_status
ORDER BY o.order_id
LIMIT 10;

## 19. Controlled repeat-run proof

`CREATE OR REPLACE` rebuilds the current Gold snapshot. To prove repeatability, compare business columns before and after rerunning the build cells.

Do not compare technical timestamps such as `_gold_created_at`; those legitimately change on a rebuild.


In [ ]:
%sql
CREATE OR REPLACE TABLE fact_order_rerun_baseline
USING DELTA AS
SELECT
  order_id, customer_region, customer_state, customer_segment, order_status,
  purchase_ts, delivered_ts, estimated_delivery_ts, return_ts,
  gmv, item_total, freight_total, payment_total,
  reconciliation_status, payment_order_difference
FROM fact_order;

**Action:** rerun the upstream fact/summarisation build cells, then execute the comparison below.

In [ ]:
%sql
WITH baseline_minus_current AS (
  SELECT * FROM fact_order_rerun_baseline
  EXCEPT
  SELECT
    order_id, customer_region, customer_state, customer_segment, order_status,
    purchase_ts, delivered_ts, estimated_delivery_ts, return_ts,
    gmv, item_total, freight_total, payment_total,
    reconciliation_status, payment_order_difference
  FROM fact_order
),
current_minus_baseline AS (
  SELECT
    order_id, customer_region, customer_state, customer_segment, order_status,
    purchase_ts, delivered_ts, estimated_delivery_ts, return_ts,
    gmv, item_total, freight_total, payment_total,
    reconciliation_status, payment_order_difference
  FROM fact_order
  EXCEPT
  SELECT * FROM fact_order_rerun_baseline
)
SELECT
  (SELECT COUNT(*) FROM baseline_minus_current) AS baseline_minus_current,
  (SELECT COUNT(*) FROM current_minus_baseline) AS current_minus_baseline,
  CASE
    WHEN (SELECT COUNT(*) FROM baseline_minus_current) = 0
     AND (SELECT COUNT(*) FROM current_minus_baseline) = 0
    THEN 'PASS' ELSE 'CHECK'
  END AS rerun_status;

In [ ]:
%sql
DROP TABLE fact_order_rerun_baseline;

## 20. Delta history / physical output checks

The Week-7 evidence must show that the Gold outputs are real Delta objects and that controlled rebuilds are traceable. Capture the actual Databricks history after execution.


In [ ]:
%sql
DESCRIBE HISTORY fact_order LIMIT 10;

In [ ]:
%sql
DESCRIBE HISTORY agg_sales_daily LIMIT 10;

In [ ]:
%sql
DESCRIBE HISTORY agg_seller_performance LIMIT 10;

In [ ]:
%sql
DESCRIBE HISTORY agg_category_sales LIMIT 10;

In [ ]:
%sql
DESCRIBE HISTORY agg_delivery_delay LIMIT 10;

In [ ]:
%sql
DESCRIBE HISTORY agg_payment_review LIMIT 10;

## 21. Week-7 acceptance checklist

Before submission, the genuine Databricks run should prove:

- [ ] Week-6 Trusted Silver handoff is complete.
- [ ] Gold reads only Trusted Silver / governed Gold facts, never Candidate or Quarantine.
- [ ] Six dimensions exist at declared grains.
- [ ] `fact_order_item` is unique at `order_item_id`.
- [ ] `fact_order` is unique at `order_id`.
- [ ] `fact_payment` is unique at `payment_id`.
- [ ] `fact_review` is unique at `review_id`.
- [ ] `fact_order_status_event` is documented as the Week-7 batch lifecycle fact and kept separate from the later streaming implementation.
- [ ] Item and payment children are independently aggregated before order-level joins.
- [ ] Payment/order reconciliation uses INR 0.05.
- [ ] Five batch summaries exist.
- [ ] All eight KPI contracts are represented with explicit denominator/exclusion/zero-denominator logic.
- [ ] Gold keys are complete and unique.
- [ ] Gold measures reconcile to the appropriate detailed population.
- [ ] One real lineage anchor is traced.
- [ ] Controlled rerun produces zero business-row differences.
- [ ] Delta history is captured.
- [ ] No Power BI export/modeling or streaming implementation is added to Week 7.


## 22. Repository/evidence boundary

Required Week-7 working artifact:

`notebooks/05_gold_aggregations.ipynb`

The approved playbook also calls for:
- `docs/gold_metrics_definition.md`
- `docs/pipeline_walkthrough.md`
- `weekly_logs/week07_log.md`
- genuine Week-7 evidence/screenshots where they add information

Do not commit fabricated counts, KPI answers, screenshots or PASS statements. The Week-7 evidence must agree with the actual Databricks execution.


## 23. Ownership and viva readiness

**Student A — order/fulfilment facts:** explain `fact_order`, delivery logic and order-level reconciliation.

**Student B — item/seller/category facts:** explain `fact_order_item`, seller/category summaries and fan-out prevention.

**Student C — payment/review facts and KPI checks:** explain `fact_payment`, `fact_review`, payment reconciliation and KPI denominators.

All three students must be able to explain the complete pipeline.

### Core viva questions
1. Why must payment installments be aggregated before joining item rows?
2. What is the grain of `fact_order_item` and `fact_order`?
3. How does the INR 0.05 reconciliation protect against fan-out?
4. Why is GMV based on item value and not freight?
5. How do zero denominators behave in rate KPIs?
6. Why must Gold read Trusted Silver rather than Candidate or Quarantine?
7. How can you prove that `fact_order` did not multiply order rows?
8. How can one dashboard number be traced back to the original source?


## 24. Final boundary

**Week 7:** Gold model, KPI logic, governed facts/summaries, reconciliation and evidence.

**Week 8:** Gold export, Power BI model and Page 1.

**Later:** streaming simulation and live-order Gold.

This notebook intentionally does not build a Power BI model, export files, streaming checkpoint, or later-week live-order output.
